# 第 4 章（Iterative Loop 版）

与书里一致，仅标 **【适配】** 处有改动。kernel 选 `Python (agentic-book)`。

默认用本地 9B（`qwen3.5-9b`）：强模型基线一次就满分，循环第 1 轮就会退出。
改 `BACKEND = "deepseek"` 可切回 DeepSeek。

本次运行里 9B 到第 3 轮时批评段退化成重复（被 2048 token 截断）——弱 critic 的典型表现，这正好是本章想让人看见的：循环能不能收敛，取决于 critic 的判断力。


In [1]:
import sys
import time
from pathlib import Path

# 【适配】找到含 common/ 的根目录
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

from langchain_core.messages import SystemMessage, HumanMessage
from common import make_llm

# --- Configuration ---
# 【适配】原书是 load_dotenv() + 检查 OPENAI_API_KEY + ChatOpenAI("gpt-4o")
BACKEND = "local"          # "local" | "deepseek"

if BACKEND == "local":
    from ch04_reflection.local_llm import make_local_llm, server_is_up
    if not server_is_up():
        raise RuntimeError("Ollama 没在跑：~/.local/ollama/bin/ollama serve &")
    llm = make_local_llm(model="qwen3.5-9b:latest", temperature=0.1,
                         num_predict=2048, num_ctx=8192)
else:
    llm = make_llm(temperature=0.1)

print("后端:", BACKEND, "| 模型:", getattr(llm, "model_name", "?"))


def run_reflection_loop():
    """
    Demonstrates a multi-step AI reflection loop to progressively improve a Python function.
    """

    # --- The Core Task ---
    task_prompt = """
    Your task is to create a Python function named `calculate_factorial`.
    This function should do the following:
    1.  Accept a single integer `n` as input.
    2.  Calculate its factorial (n!).
    3.  Include a clear docstring explaining what the function does.
    4.  Handle edge cases: The factorial of 0 is 1.
    5.  Handle invalid input: Raise a ValueError if the input is a negative number.
    6.  The function must also work for large inputs such as n = 5000.
    """

    # --- The Reflection Loop ---
    max_iterations = 3
    current_code = ""
    message_history = [HumanMessage(content=task_prompt)]

    for i in range(max_iterations):
        print("\n" + "="*25 + f" REFLECTION LOOP: ITERATION {i + 1} " + "="*25)

        # --- 1. GENERATE / REFINE STAGE ---
        if i == 0:
            print("\n>>> STAGE 1: GENERATING initial code...")
            response = llm.invoke(message_history)
            current_code = response.content
        else:
            print("\n>>> STAGE 1: REFINING code based on previous critique...")
            message_history.append(HumanMessage(content="Please refine the code using the critiques provided."))
            response = llm.invoke(message_history)
            current_code = response.content

        print("\n--- Generated Code (v" + str(i + 1) + ") ---\n" + current_code)
        message_history.append(response)

        # --- 2. REFLECT STAGE ---
        print("\n>>> STAGE 2: REFLECTING on the generated code...")

        reflector_prompt = [
            SystemMessage(content="""
                You are a senior software engineer and an expert in Python.
                Your role is to perform a meticulous code review.
                Critically evaluate the provided Python code based on the original task requirements.
                Look for bugs, style issues, missing edge cases, and areas for improvement.
                If the code is perfect and meets all requirements, respond with the single phrase 'CODE_IS_PERFECT'.
                Otherwise, provide a bulleted list of your critiques.
            """),
            HumanMessage(content=f"Original Task:\n{task_prompt}\n\nCode to Review:\n{current_code}")
        ]

        critique_response = llm.invoke(reflector_prompt)
        critique = critique_response.content

        # --- 3. STOPPING CONDITION ---
        # 注：这里保持书里的子串匹配。它有个小坑——模型说
        #     "the code is NOT CODE_IS_PERFECT" 也会命中。见 README 的坑清单。
        if "CODE_IS_PERFECT" in critique:
            print("\n--- Critique ---\nNo further critiques found. The code is satisfactory.")
            break

        print("\n--- Critique ---\n" + critique)
        message_history.append(HumanMessage(content=f"Critique of the previous code:\n{critique}"))

    print("\n" + "="*30 + " FINAL RESULT " + "="*30)
    print("\nFinal refined code after the reflection process:\n")
    print(current_code)


run_reflection_loop()

后端: local | 模型: qwen3.5-9b:latest

========================= REFLECTION LOOP: ITERATION 1 =========================

>>> STAGE 1: GENERATING initial code...



--- Generated Code (v1) ---
```python
def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By definition, 0! = 1.

    Parameters:
        n (int): A non-negative integer whose factorial is to be computed.

    Returns:
        int: The factorial of n.

    Raises:
        ValueError: If n is negative.

    Examples:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(5000)  # Works for large inputs
        127... (a very large integer)
    """
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")

    # Python's math.factorial is efficient and handles large integers automatically.
    # For educational purposes, an iterative implementation is also shown below.
    # Using math.factorial for production due to its optimiza


--- Critique ---
- **Redundant/Confusing Comments**: The comments mention an "iterative implementation is also shown below" but no iterative implementation is provided. This is misleading and should be removed or the code updated to match the comment.
- **Unnecessary Import Inside Function**: The `import math` statement is placed inside the function body. While it works, it is inefficient and considered poor practice in Python. Imports should be placed at the top of the file (module level) to be executed only once when the module is loaded.
- **Missing Type Hint for Negative Check**: While the type hint `n: int` is present, the function does not explicitly validate if the input is actually an integer (e.g., it would accept a float like `5.0` without raising an error, though `math.factorial` might handle it or raise a different error). Adding an explicit `isinstance(n, int)` check would make the validation stricter and clearer according to the requirement "Accept a single integer".
- *


--- Generated Code (v2) ---
```python
import math
from typing import Union

def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By definition, 0! = 1.

    Parameters:
        n (int): A non-negative integer whose factorial is to be computed.

    Returns:
        int: The factorial of n.

    Raises:
        TypeError: If n is not an integer.
        ValueError: If n is negative.

    Examples:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(5000)
        127... (a very large integer)
    """
    # Strict type checking to ensure input is an integer
    if not isinstance(n, int):
        raise TypeError(f"Expected an integer, got {type(n).__name__}")

    # Validate non-negative constraint
    if n < 0:
        raise ValueError("Factorial is not defin


--- Critique ---
- **Missing `Union` usage**: The function signature imports `Union` from `typing` but does not use it. While the current signature `n: int` is valid, the import is unnecessary and should be removed to avoid confusion.
- **Redundant manual validation**: The code manually checks `isinstance(n, int)` and `n < 0` before calling `math.factorial`. Since `math.factorial` already raises `TypeError` for non-integers and `ValueError` for negative numbers, these manual checks are redundant. Removing them simplifies the code and relies on the standard library's robust error handling.
- **Inconsistent error messages**: The manual `TypeError` message includes the type name, but `math.factorial` raises a generic `TypeError`. While custom messages can be helpful, they should be consistent with the function's documented behavior. If the goal is to provide clearer error messages, the manual checks are justified, but the redundancy remains.
- **Docstring example inconsistency**: The exa


--- Generated Code (v3) ---
```python
import math

def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By definition, 0! = 1.

    Parameters:
        n (int): A non-negative integer whose factorial is to be computed.

    Returns:
        int: The factorial of n.

    Raises:
        TypeError: If n is not an integer.
        ValueError: If n is negative.

    Examples:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(5000)
        ...
    """
    return math.factorial(n)
```

>>> STAGE 2: REFLECTING on the generated code...



--- Critique ---
- **Missing input type validation**: The function does not explicitly check if `n` is an integer. While `math.factorial` will raise a `TypeError` for non-integer types (e.g., floats), the docstring promises a `TypeError` for non-integers, but the implementation relies on the library's behavior without explicit validation. This is acceptable in some contexts, but for a robust function, explicit type checking is often preferred to provide clearer error messages.

- **No explicit handling of large inputs**: While `math.factorial` can handle large inputs like `n = 5000`, the function does not explicitly mention or handle potential memory or performance issues for extremely large values. However, this is more of a documentation concern than a code issue.

- **Redundant import**: The `import math` statement is unnecessary if the function is intended to be self-contained and not rely on external libraries. While using `math.factorial` is efficient, the task does not explicit